# Fine-tune dense encoder & cross-encoder trên Colab (NLP501 · Nhóm 2)

Notebook này chạy phần **huấn luyện** của dự án trên GPU miễn phí của Colab. Không cần tải repo, cũng không cần dữ liệu trên máy:

| Mô hình | Gốc | Cách huấn luyện | Dùng trong |
|---|---|---|---|
| Dense encoder | `intfloat/multilingual-e5-base` | Contrastive (CachedMultipleNegativesRankingLoss), triplet (JD, Good Fit, No Fit), in-batch negative, hướng ConFit | B1, B2 và retrieval của P6 |
| Cross-encoder | `BAAI/bge-reranker-base` | Binary cross-entropy, nhãn No Fit / Potential / Good = 0 / 0.5 / 1 | B4, P6 (rerank) |

**Dữ liệu:** `cnamuangtoun/resume-job-description-fit` được tải thẳng từ Hugging Face. Không có CV nào trên máy nhóm được gửi lên (NFR-02).

**Chống rò rỉ:** bộ dữ liệu này cũng là tập kiểm tra thật của nhóm (`data/public`), nên:
1. Bỏ mọi JD QA / Automation Test (đây chính là các JD dùng để đánh giá).
2. Bỏ mọi CV từng được gán nhãn cho một JD như vậy.
3. Chia `train.csv` thành train / dev theo JD. `test.csv` (đã lọc như trên, bỏ cả JD đã gặp) chỉ dùng để kiểm tra cuối.

**Cách chạy:** chọn *Runtime → Change runtime type → T4 GPU*, rồi chọn *Runtime → Run all*. Ô cài đặt nâng cấp `sentence-transformers` / `transformers`. Nếu Colab báo cần khởi động lại, hoặc ô huấn luyện lỗi import, hãy chọn *Runtime → Restart session* rồi *Run all* lần nữa (lần hai không cài lại). Với T4, mỗi mô hình mất khoảng 10–20 phút cho 1 epoch. Ô cuối nén hai mô hình thành `rjf_finetuned_models.zip` để tải về.

> File này được sinh tự động từ `cv_agent/training/finetune.py` bằng `python scripts/build_colab_notebook.py`. Đừng sửa tay: hãy sửa module rồi sinh lại.

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "Không có GPU: vẫn chạy được nhưng chậm.")
!pip -q install "sentence-transformers>=4.0" "datasets>=2.19" "accelerate>=0.26" pyyaml

## Mã nguồn dự án dùng cho huấn luyện
Các ô dưới đây ghi nguyên văn các module của repo.

In [ ]:
import os
for d in ["cv_agent", "cv_agent/data", "cv_agent/eval", "cv_agent/training"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile cv_agent/__init__.py
"""Agentic CV search for Automation Test roles (NLP501 - Nhom 2)."""
__version__ = "0.1.0"

In [ ]:
%%writefile cv_agent/data/__init__.py

In [ ]:
%%writefile cv_agent/data/public.py
"""Public datasets as a real-data test bed (D2), in the same layout as ``data/synthetic``.

* Resume–Job Fit (``cnamuangtoun/resume-job-description-fit`` on Hugging Face): real resume / job description
  pairs labelled No Fit / Potential Fit / Good Fit. Only QA / test-automation job descriptions are kept.
  The dataset card states no licence: use it for course research only and do not redistribute it
  (``data/public/`` is git-ignored).
* SkillSpan (``jjzha/skillspan``, CC-BY-4.0): job postings with skill / knowledge spans, used to measure
  span-level F1 of the entity extractor (FR-03), see ``cv_agent.eval.span_metrics``.

Each JD is labelled only against ~25 resumes, so ``gold.json`` sets ``judged_only`` and metrics are computed
on the labelled resumes of each JD (condensed-list evaluation), never treating unlabelled CVs as irrelevant.
"""
from __future__ import annotations

import csv
import hashlib
import json
import logging
import re
import sys
import unicodedata
import urllib.request
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import yaml

log = logging.getLogger(__name__)

HF_BASE = "https://huggingface.co/datasets"
RJF_REPO = "cnamuangtoun/resume-job-description-fit"
RJF_FILES = ("train.csv", "test.csv")
SKILLSPAN_REPO = "jjzha/skillspan"
SKILLSPAN_FILES = ("dev.json", "test.json")
LABELS = {"No Fit": 0, "Potential Fit": 1, "Good Fit": 2}
# Strong QA signals; a JD is kept when it contains at least MIN_QA_SIGNALS *different* ones. A single loose
# mention ("QA", "test cases") also matches business-analyst or electrical-engineer postings.
QA_SIGNAL_RE = re.compile(
    r"\b(?:test automation|automation test\w*|automated test\w*|QA (?:engineer|analyst|automation|tester)|SDET|"
    r"software engineer in test|selenium|cypress|playwright|appium|cucumber|testng|test cases?|test plans?|"
    r"quality assurance (?:engineer|analyst)|software test\w*|regression test\w*)\b", re.IGNORECASE)
MIN_QA_SIGNALS = 2
DEV_SHARE = 0.3          # share of JDs used to tune thresholds (split=dev), chosen by stable hash
DOWNLOAD_TIMEOUT_S = 120
DEFAULT_PROFILE = "Mặc định"


def download(repo: str, filename: str, raw_dir: Path) -> Path:
    """Download one dataset file once (kept in ``raw_dir``); later calls reuse the local copy."""
    raw_dir.mkdir(parents=True, exist_ok=True)
    target = raw_dir / f"{repo.replace('/', '__')}__{filename}"
    if target.exists() and target.stat().st_size > 0:
        return target
    url = f"{HF_BASE}/{repo}/resolve/main/{filename}"
    log.info("downloading %s", url)
    tmp = target.with_suffix(target.suffix + ".part")
    with urllib.request.urlopen(url, timeout=DOWNLOAD_TIMEOUT_S) as resp, open(tmp, "wb") as fh:  # noqa: S310 (fixed https host)
        expected = int(resp.headers.get("Content-Length") or 0)
        while chunk := resp.read(1 << 20):
            fh.write(chunk)
    size = tmp.stat().st_size
    if size == 0 or (expected and size != expected):
        tmp.unlink()
        raise OSError(f"incomplete download of {url}: {size} of {expected or '?'} bytes")
    tmp.replace(target)
    return target


def _stable_id(prefix: str, text: str) -> str:
    return f"{prefix}_{hashlib.sha1(text.encode('utf-8')).hexdigest()[:10]}"


def _title(jd_text: str) -> str:
    first = next((line.strip() for line in jd_text.splitlines() if line.strip()), "Job description")
    return first[:80]


def read_pairs(paths: Iterable[Path]) -> List[Tuple[str, str, int]]:
    """(resume_text, job_description_text, label 0..2) rows; rows with an unknown label are skipped."""
    csv.field_size_limit(min(sys.maxsize, 2**31 - 1))
    rows = []
    for path in paths:
        with open(path, encoding="utf-8", newline="") as fh:
            for row in csv.DictReader(fh):
                label = LABELS.get((row.get("label") or "").strip())
                resume, jd = (row.get("resume_text") or "").strip(), (row.get("job_description_text") or "").strip()
                if label is not None and resume and jd:
                    rows.append((unicodedata.normalize("NFC", resume), unicodedata.normalize("NFC", jd), label))
    return rows


def qa_signals(jd_text: str) -> List[str]:
    return sorted({m.group(0).lower() for m in QA_SIGNAL_RE.finditer(jd_text)})


def build_resume_job_fit(pairs: List[Tuple[str, str, int]], out_dir: Path,
                         min_signals: int = MIN_QA_SIGNALS) -> Dict:
    """Write cvs/, jds.yaml, gold.json and meta.json for the QA / test-automation job descriptions."""
    qa = [(r, j, lab) for r, j, lab in pairs if len(qa_signals(j)) >= min_signals]
    if not qa:
        raise ValueError(f"no job description has {min_signals} or more QA signals")
    cv_dir = out_dir / "cvs"
    cv_dir.mkdir(parents=True, exist_ok=True)
    for stale in cv_dir.glob("rjf_*.txt"):  # a rebuild with another filter must not keep old CVs
        stale.unlink()
    gold: Dict[str, Dict] = {}
    jd_texts: Dict[str, str] = {}
    for resume, jd_text, label in qa:
        cv_id, jd_id = _stable_id("rjf", resume), _stable_id("rjf_jd", jd_text)
        (cv_dir / f"{cv_id}.txt").write_text(resume, encoding="utf-8")
        jd_texts[jd_id] = jd_text
        rel = gold.setdefault(jd_id, {"relevance": {}})["relevance"]
        rel[cv_id] = max(label, rel.get(cv_id, 0))  # duplicate pair: keep the higher label
    ordered = sorted(jd_texts, key=lambda j: hashlib.sha1(j.encode()).hexdigest())
    n_dev = max(1, round(len(ordered) * DEV_SHARE)) if len(ordered) > 1 else 0
    specs = []
    for i, jd_id in enumerate(ordered):
        split = "dev" if i < n_dev else "test"
        rel = gold[jd_id]["relevance"]
        gold[jd_id].update({"hr_ranking": sorted(rel, key=lambda c: (-rel[c], c)), "split": split,
                            "ambiguous": False, "judged_only": True})
        specs.append({"id": jd_id, "title": _title(jd_texts[jd_id]), "company_type": "default",
                      "profile": DEFAULT_PROFILE, "split": split, "qa_signals": qa_signals(jd_texts[jd_id]),
                      "text": jd_texts[jd_id]})
    meta = {"synthetic": False, "source": f"{HF_BASE}/{RJF_REPO}", "license": "not stated on the dataset card",
            "labels": LABELS, "qa_signal_pattern": QA_SIGNAL_RE.pattern, "min_qa_signals": min_signals, "n_pairs": len(qa),
            "n_cvs": len({c for g in gold.values() for c in g["relevance"]}), "n_jds": len(gold)}
    (out_dir / "jds.yaml").write_text(yaml.safe_dump(specs, allow_unicode=True, sort_keys=False, width=120),
                                      encoding="utf-8")
    (out_dir / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=1), encoding="utf-8")
    (out_dir / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
    return meta


def fetch_resume_job_fit(out_dir: Path, raw_dir: Path, min_signals: int = MIN_QA_SIGNALS) -> Dict:
    return build_resume_job_fit(read_pairs(download(RJF_REPO, f, raw_dir) for f in RJF_FILES), out_dir, min_signals)


def fetch_skillspan(raw_dir: Path) -> List[Path]:
    return [download(SKILLSPAN_REPO, f, raw_dir) for f in SKILLSPAN_FILES]

In [ ]:
%%writefile cv_agent/eval/__init__.py

In [ ]:
%%writefile cv_agent/eval/metrics.py
"""Ranking metrics, agreement and paired statistics (D5)."""
from __future__ import annotations

import math
import random
from typing import Dict, List, Sequence, Tuple


def recall_at_k(ranking: Sequence[str], rel: Dict[str, int], k: int, min_grade: int = 1) -> float:
    relevant = {c for c, g in rel.items() if g >= min_grade}
    if not relevant:
        return float("nan")
    return len(relevant & set(ranking[:k])) / len(relevant)


def mrr_at_k(ranking: Sequence[str], rel: Dict[str, int], k: int = 10, min_grade: int = 2) -> float:
    for i, c in enumerate(ranking[:k], start=1):
        if rel.get(c, 0) >= min_grade:
            return 1.0 / i
    return 0.0


def ndcg_at_k(ranking: Sequence[str], rel: Dict[str, int], k: int = 10) -> float:
    dcg = sum((2 ** rel.get(c, 0) - 1) / math.log2(i + 2) for i, c in enumerate(ranking[:k]))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg else float("nan")


def kendall_tau(system: Sequence[str], hr: Sequence[str]) -> float:
    """Kendall τ-b between HR order and the system's order over HR-ranked items.

    Items the system did not return are tied at the bottom of the system order.
    """
    pos = {c: i for i, c in enumerate(system)}
    bottom = len(system)
    sys_rank = [pos.get(c, bottom) for c in hr]
    n = len(hr)
    if n < 2:
        return float("nan")
    conc = disc = ties_sys = 0
    for i in range(n):
        for j in range(i + 1, n):
            d = sys_rank[i] - sys_rank[j]
            if d == 0:
                ties_sys += 1
            elif d < 0:
                conc += 1
            else:
                disc += 1
    n0 = n * (n - 1) / 2
    denom = math.sqrt(n0 * (n0 - ties_sys))
    return (conc - disc) / denom if denom else 0.0


def cohen_kappa(a: Dict[str, int], b: Dict[str, int]) -> float:
    keys = sorted(set(a) & set(b))
    if not keys:
        return float("nan")
    cats = sorted({a[k] for k in keys} | {b[k] for k in keys})
    n = len(keys)
    po = sum(a[k] == b[k] for k in keys) / n
    pe = sum((sum(a[k] == c for k in keys) / n) * (sum(b[k] == c for k in keys) / n) for c in cats)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


# ---------------------------------------------------------------- paired statistics over JDs

def _clean(x: Sequence[float], y: Sequence[float]) -> List[Tuple[float, float]]:
    return [(a, b) for a, b in zip(x, y) if not (math.isnan(a) or math.isnan(b))]


def paired_bootstrap(x: Sequence[float], y: Sequence[float], n_boot: int = 10_000, seed: int = 42,
                     alpha: float = 0.05) -> Dict[str, float]:
    """Mean difference x - y with percentile CI, resampling JDs (paired)."""
    pairs = _clean(x, y)
    if not pairs:
        return {"mean_diff": float("nan"), "ci_low": float("nan"), "ci_high": float("nan"), "n": 0}
    diffs = [a - b for a, b in pairs]
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(diffs) for _ in diffs) / len(diffs) for _ in range(n_boot))
    lo, hi = boots[int(alpha / 2 * n_boot)], boots[int((1 - alpha / 2) * n_boot) - 1]
    return {"mean_diff": sum(diffs) / len(diffs), "ci_low": lo, "ci_high": hi, "n": len(diffs)}


def wilcoxon_signed_rank(x: Sequence[float], y: Sequence[float]) -> Dict[str, float]:
    """Two-sided Wilcoxon signed-rank (normal approximation; exact via scipy if installed)."""
    diffs = [a - b for a, b in _clean(x, y) if a - b != 0]
    n = len(diffs)
    if n == 0:
        return {"W": 0.0, "p_value": 1.0, "n": 0, "effect_size_r": 0.0}
    try:
        from scipy.stats import wilcoxon
        res = wilcoxon(diffs)
        p = float(res.pvalue)
    except ImportError:
        p = None
    order = sorted(range(n), key=lambda i: abs(diffs[i]))
    ranks = [0.0] * n
    i = 0
    while i < n:  # average ranks for ties
        j = i
        while j + 1 < n and abs(diffs[order[j + 1]]) == abs(diffs[order[i]]):
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2 + 1
        i = j + 1
    w_plus = sum(r for r, d in zip(ranks, diffs) if d > 0)
    mean, sd = n * (n + 1) / 4, math.sqrt(n * (n + 1) * (2 * n + 1) / 24)
    z = (w_plus - mean) / sd if sd else 0.0
    if p is None:
        p = math.erfc(abs(z) / math.sqrt(2))
    return {"W": w_plus, "p_value": p, "n": n, "effect_size_r": z / math.sqrt(n)}


def graded_tau(system: Sequence[str], relevance: Dict[str, int]) -> float:
    """Kendall-style agreement with graded labels: only pairs with *different* grades count, so the arbitrary
    order inside one grade (e.g. two "Good Fit" CVs) is not penalised. (concordant - discordant) / both."""
    items = [c for c in system if c in relevance]
    conc = disc = 0
    for i, a in enumerate(items):
        for b in items[i + 1:]:
            if relevance[a] > relevance[b]:
                conc += 1
            elif relevance[a] < relevance[b]:
                disc += 1
    return (conc - disc) / (conc + disc) if conc + disc else float("nan")


In [ ]:
%%writefile cv_agent/training/__init__.py
"""Optional ConFit-style fine-tuning of the dense encoder and the cross-encoder (runs best on a GPU / Colab)."""

In [ ]:
%%writefile cv_agent/training/finetune.py
"""Fine-tune the B1 dense encoder and the B4 cross-encoder on public resume–job pairs (ConFit-style, optional).

Data: ``cnamuangtoun/resume-job-description-fit`` (Hugging Face), downloaded directly — no local CV leaves the
machine (NFR-02). Leakage rules, because the same dataset is our real-data test bed (``data/public``):

* every pair whose job description is a QA / test-automation JD is dropped (those are the evaluation JDs);
* every resume that was judged for such a JD — in either CSV file — is dropped from all other pairs too;
* the remaining ``train.csv`` pairs are split by job description into train / dev; ``test.csv`` (also filtered)
  is a held-out check that is never used for early stopping or model selection.

Dense encoder: (Cached)MultipleNegativesRankingLoss on (JD, Good Fit resume, No Fit resume of the same JD) triplets,
in-batch negatives without duplicates — the contrastive recipe of ConFit, without its data augmentation.
Cross-encoder: binary cross-entropy on (JD, resume) with labels No Fit / Potential / Good = 0 / 0.5 / 1.

The same code runs locally (``python -m cv_agent finetune``) and in ``notebooks/finetune_colab.ipynb``, which is
generated from this file by ``scripts/build_colab_notebook.py``.
"""
from __future__ import annotations

import hashlib
import json
import logging
import time
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Set, Tuple

from cv_agent.data.public import MIN_QA_SIGNALS, RJF_REPO, download, qa_signals, read_pairs
from cv_agent.eval.metrics import ndcg_at_k

log = logging.getLogger(__name__)

Row = Tuple[str, str, int]            # (resume_text, job_description_text, label 0..2)
GOOD, NO_FIT, MAX_LABEL = 2, 0, 2
TARGETS = ("dense", "rerank")
DEFAULT_BASE = {"dense": "intfloat/multilingual-e5-base", "rerank": "BAAI/bge-reranker-base"}
DEV_SHARE = 0.1
EVAL_K = 10
RERANK_INFERENCE_LEN = 512   # same as cv_agent.ranking.reranker.CrossEncoderReranker


@dataclass(frozen=True)
class TrainConfig:
    target: str                       # dense | rerank
    base_model: str
    out_dir: Path
    epochs: int = 1
    batch_size: int = 16              # in-batch negatives per step (ConFit-style contrastive loss)
    mini_batch_size: int = 4          # forward chunk of the cached loss: same gradients, far less memory
    learning_rate: float = 2e-5
    max_len: int = 256
    warmup_ratio: float = 0.1
    seed: int = 42
    limit: Optional[int] = None       # cap training examples / eval JDs for a quick smoke run

    def __post_init__(self) -> None:
        if self.target not in TARGETS:
            raise ValueError(f"target must be one of {TARGETS}, got {self.target!r}")
        if self.epochs < 1 or self.batch_size < 2 or self.max_len < 32 or not 0 < self.learning_rate < 1:
            raise ValueError("epochs >= 1, batch_size >= 2, max_len >= 32 and 0 < learning_rate < 1 are required")
        if not 1 <= self.mini_batch_size <= self.batch_size:
            raise ValueError("1 <= mini_batch_size <= batch_size is required")


# ------------------------------------------------------------------ data (pure, unit-tested)
def is_qa_jd(jd_text: str) -> bool:
    return len(qa_signals(jd_text)) >= MIN_QA_SIGNALS


def eval_resumes(rows: Sequence[Row]) -> Set[str]:
    return {r for r, jd, _ in rows if is_qa_jd(jd)}


def leakage_safe_rows(rows: Sequence[Row], judged: Optional[Set[str]] = None) -> List[Row]:
    """Drop QA JDs (our evaluation JDs) and every resume judged for one of them.

    ``judged`` must cover every file the evaluation set is built from (``data/public`` uses train + test).
    """
    eval_resumes_ = eval_resumes(rows) if judged is None else judged
    return [(r, jd, lab) for r, jd, lab in rows if not is_qa_jd(jd) and r not in eval_resumes_]


def _jd_key(jd_text: str) -> str:
    return hashlib.sha1(jd_text.encode("utf-8")).hexdigest()


def split_by_jd(rows: Sequence[Row], dev_share: float = DEV_SHARE) -> Tuple[List[Row], List[Row]]:
    """Group split: a JD and all its pairs land in exactly one split (stable hash order, input order ignored)."""
    jds = sorted({jd for _, jd, _ in rows}, key=_jd_key)
    n_dev = max(1, round(len(jds) * dev_share)) if len(jds) > 1 else 0
    dev_jds = set(jds[:n_dev])
    return ([row for row in rows if row[1] not in dev_jds], [row for row in rows if row[1] in dev_jds])


def triplets(rows: Sequence[Row]) -> List[Tuple[str, str, str]]:
    """(JD, Good Fit resume, No Fit resume of the same JD); negatives are reused round-robin."""
    good: Dict[str, List[str]] = defaultdict(list)
    bad: Dict[str, List[str]] = defaultdict(list)
    for resume, jd, label in rows:
        if label == GOOD:
            good[jd].append(resume)
        elif label == NO_FIT:
            bad[jd].append(resume)
    out = []
    for jd in sorted(good, key=_jd_key):
        negatives = sorted(bad.get(jd, []))
        for i, pos in enumerate(sorted(good[jd])):
            if negatives:
                out.append((jd, pos, negatives[i % len(negatives)]))
    return out


def cross_pairs(rows: Sequence[Row]) -> List[Tuple[str, str, float]]:
    return [(jd, resume, label / MAX_LABEL) for resume, jd, label in rows]


def with_prefix(texts: Sequence[str], kind: str, model_name: str) -> List[str]:
    """multilingual-e5 expects 'query: ' / 'passage: ' (same rule as SentenceTransformerEncoder)."""
    return [f"{kind}: {t}" for t in texts] if is_e5(model_name) else list(texts)


def is_e5(model_name: str) -> bool:
    """Judge by the model / folder name only, never by parent folders ('runs/e5-x/bge-m3-ft' is not e5)."""
    return "e5" in Path(model_name).name


ScoreFn = Callable[[str, List[str]], Sequence[float]]


def ndcg_by_jd(score_fn: ScoreFn, rows: Sequence[Row], k: int = EVAL_K) -> Dict[str, float]:
    """Mean nDCG@k over JDs with >= 2 judged resumes and >= 1 relevant one (condensed-list evaluation)."""
    by_jd: Dict[str, Dict[str, int]] = defaultdict(dict)
    for resume, jd, label in rows:
        by_jd[jd][resume] = max(label, by_jd[jd].get(resume, 0))
    scores = []
    for jd in sorted(by_jd, key=_jd_key):
        rel = by_jd[jd]
        if len(rel) < 2 or not any(rel.values()):
            continue
        resumes = sorted(rel)
        s = score_fn(jd, resumes)
        ranking = [r for _, r in sorted(zip(s, resumes), key=lambda p: (-p[0], p[1]))]
        scores.append(ndcg_at_k(ranking, rel, k))
    return {f"ndcg@{k}": sum(scores) / len(scores) if scores else float("nan"), "n_jds": len(scores)}


def _limit_jds(rows: Sequence[Row], limit: Optional[int]) -> List[Row]:
    if not limit:
        return list(rows)
    keep = set(sorted({jd for _, jd, _ in rows}, key=_jd_key)[:limit])
    return [row for row in rows if row[1] in keep]


# ------------------------------------------------------------------ models (needs requirements-ml.txt)
def _device() -> str:
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def _dense_scorer(model, model_name: str) -> ScoreFn:
    def score(jd: str, resumes: List[str]) -> List[float]:
        q = model.encode(with_prefix([jd], "query", model_name), normalize_embeddings=True)
        d = model.encode(with_prefix(resumes, "passage", model_name), normalize_embeddings=True, batch_size=32)
        return (d @ q[0]).tolist()
    return score


def _rerank_scorer(model) -> ScoreFn:
    def score(jd: str, resumes: List[str]) -> List[float]:
        return [float(x) for x in model.predict([(jd, r) for r in resumes], batch_size=32)]
    return score


def _training_args(cls, cfg: TrainConfig, **extra):
    fp16 = _device() == "cuda"
    return cls(output_dir=str(cfg.out_dir / "checkpoints"), num_train_epochs=cfg.epochs,
               per_device_train_batch_size=cfg.batch_size, learning_rate=cfg.learning_rate,
               warmup_ratio=cfg.warmup_ratio, fp16=fp16, seed=cfg.seed, save_strategy="no", logging_steps=20,
               report_to="none", **extra)


def train_dense(cfg: TrainConfig, train_rows: Sequence[Row], eval_sets: Dict[str, Sequence[Row]]) -> Dict:
    from datasets import Dataset
    from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                       SentenceTransformerTrainingArguments, losses)
    from sentence_transformers.training_args import BatchSamplers

    model = SentenceTransformer(cfg.base_model, device=_device())
    inference_len = model.max_seq_length  # evaluate and save at the length the pipeline uses (512 for e5)
    trips = triplets(train_rows)[: cfg.limit or None]
    before = {name: ndcg_by_jd(_dense_scorer(model, cfg.base_model), rows) for name, rows in eval_sets.items()}
    model.max_seq_length = cfg.max_len  # shorter only while training, for speed
    data = Dataset.from_dict({"anchor": with_prefix([t[0] for t in trips], "query", cfg.base_model),
                              "positive": with_prefix([t[1] for t in trips], "passage", cfg.base_model),
                              "negative": with_prefix([t[2] for t in trips], "passage", cfg.base_model)})
    args = _training_args(SentenceTransformerTrainingArguments, cfg, batch_sampler=BatchSamplers.NO_DUPLICATES)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=data,
                               loss=losses.CachedMultipleNegativesRankingLoss(
                                   model, mini_batch_size=cfg.mini_batch_size)).train()
    model.max_seq_length = inference_len
    model.save(str(cfg.out_dir))
    after = {name: ndcg_by_jd(_dense_scorer(model, cfg.base_model), rows) for name, rows in eval_sets.items()}
    return {"n_train_examples": len(trips), "before": before, "after": after}


def train_rerank(cfg: TrainConfig, train_rows: Sequence[Row], eval_sets: Dict[str, Sequence[Row]]) -> Dict:
    from datasets import Dataset
    from sentence_transformers.cross_encoder import (CrossEncoder, CrossEncoderTrainer,
                                                     CrossEncoderTrainingArguments)
    from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

    model = CrossEncoder(cfg.base_model, num_labels=1, max_length=RERANK_INFERENCE_LEN, device=_device())
    pairs = cross_pairs(train_rows)[: cfg.limit or None]
    before = {name: ndcg_by_jd(_rerank_scorer(model), rows) for name, rows in eval_sets.items()}
    model.max_length = cfg.max_len  # shorter only while training, for speed
    data = Dataset.from_dict({"query": [p[0] for p in pairs], "passage": [p[1] for p in pairs],
                              "label": [p[2] for p in pairs]})
    CrossEncoderTrainer(model=model, args=_training_args(CrossEncoderTrainingArguments, cfg), train_dataset=data,
                        loss=BinaryCrossEntropyLoss(model)).train()
    model.max_length = RERANK_INFERENCE_LEN
    model.save(str(cfg.out_dir))
    after = {name: ndcg_by_jd(_rerank_scorer(model), rows) for name, rows in eval_sets.items()}
    return {"n_train_examples": len(pairs), "before": before, "after": after}


def split_files(train_csv: Sequence[Row], test_csv: Sequence[Row]) -> Dict[str, List[Row]]:
    """Leakage-safe train / dev / held-out test rows; evaluation resumes are collected over BOTH files."""
    judged = eval_resumes(list(train_csv) + list(test_csv))
    train_all, test_all = leakage_safe_rows(train_csv, judged), leakage_safe_rows(test_csv, judged)
    train, dev = split_by_jd(train_all)
    seen = {jd for _, jd, _ in train_all}
    return {"train": train, "dev": dev, "test": [row for row in test_all if row[1] not in seen]}


def load_rows(raw_dir: Path) -> Dict[str, List[Row]]:
    return split_files(read_pairs([download(RJF_REPO, "train.csv", raw_dir)]),
                       read_pairs([download(RJF_REPO, "test.csv", raw_dir)]))


def run(cfg: TrainConfig, raw_dir: Path) -> Dict:
    """Train one target, save the model to ``cfg.out_dir`` and write ``finetune_metrics.json`` next to it."""
    started = time.time()
    rows = load_rows(raw_dir)
    eval_sets = {"dev": _limit_jds(rows["dev"], cfg.limit), "test": _limit_jds(rows["test"], cfg.limit)}
    cfg.out_dir.mkdir(parents=True, exist_ok=True)
    trainer = train_dense if cfg.target == "dense" else train_rerank
    result = trainer(cfg, rows["train"], eval_sets)
    report = {"config": {**asdict(cfg), "out_dir": str(cfg.out_dir)}, "device": _device(),
              "data": {name: {"pairs": len(r), "jds": len({jd for _, jd, _ in r})} for name, r in rows.items()},
              **result, "wall_s": round(time.time() - started, 1)}
    (cfg.out_dir / "finetune_metrics.json").write_text(json.dumps(report, indent=1), encoding="utf-8")
    log.info("fine-tuned %s -> %s: %s", cfg.target, cfg.out_dir, result["after"])
    return report

## Tham số

In [ ]:
# Tham số huấn luyện. LIMIT = None: chạy đủ dữ liệu; đặt LIMIT = 64 để chạy thử vài phút.
TARGETS = ["dense", "rerank"]
EPOCHS = 1
BATCH_SIZE = 16
MAX_LEN = 256          # 512 tốt hơn một chút nhưng chậm gấp ~3 lần
LIMIT = None
OUT_DIR = "models"
RAW_DIR = "data/public/raw"

## Huấn luyện và đánh giá trước / sau

In [ ]:
import logging
from pathlib import Path
from cv_agent.training.finetune import DEFAULT_BASE, TrainConfig, run

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
reports = {}
for target in TARGETS:
    base = DEFAULT_BASE[target]
    cfg = TrainConfig(target=target, base_model=base, out_dir=Path(OUT_DIR) / f"{base.split('/')[-1]}-rjf-ft",
                      epochs=EPOCHS, batch_size=BATCH_SIZE, max_len=MAX_LEN, limit=LIMIT)
    reports[target] = run(cfg, Path(RAW_DIR))

In [ ]:
import pandas as pd

rows = []
for target, rep in reports.items():
    for split in rep["after"]:
        rows.append({"mô hình": target, "tập": split, "số JD": rep["after"][split]["n_jds"],
                     "nDCG@10 trước": round(rep["before"][split]["ndcg@10"], 3),
                     "nDCG@10 sau": round(rep["after"][split]["ndcg@10"], 3),
                     "ví dụ train": rep["n_train_examples"], "thời gian (s)": rep["wall_s"]})
pd.DataFrame(rows)

## Tải mô hình về

In [ ]:
import shutil
shutil.rmtree(Path(OUT_DIR) / "multilingual-e5-base-rjf-ft" / "checkpoints", ignore_errors=True)
shutil.rmtree(Path(OUT_DIR) / "bge-reranker-base-rjf-ft" / "checkpoints", ignore_errors=True)
archive = shutil.make_archive("rjf_finetuned_models", "zip", OUT_DIR)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass  # not on Colab: the zip is next to the notebook

## Dùng mô hình đã fine-tune trên máy

Giải nén `rjf_finetuned_models.zip` vào thư mục `models/` của dự án, rồi chạy lại thực nghiệm. Chỉ cần đổi mô hình, mọi cấu hình còn lại giữ nguyên:

```bash
python -m cv_agent experiments --neural --llm ollama --model qwen2.5:3b --configs B0,B1,B2,B3,B4,P6 \
    --dense-model models/multilingual-e5-base-rjf-ft --rerank-model models/bge-reranker-base-rjf-ft
python -m cv_agent experiments --data data/public --neural --llm ollama --model qwen2.5:3b --configs B0,B1,B2,B3,B4,P6 \
    --dense-model models/multilingual-e5-base-rjf-ft --rerank-model models/bge-reranker-base-rjf-ft
```

`finetune_metrics.json` trong mỗi thư mục mô hình ghi lại cấu hình, số cặp train / dev / test và nDCG@10 trước / sau.